In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

# =========================================================
# STL-BASED 3-YEAR FORECAST WITH CI
# Works with:
#   mlp_results["monthly_pr"]
#   xgb_results["monthly_pr"]
# from the earlier code
# =========================================================

def stl_forecast_with_ci(monthly_series, horizon=36, seasonal_period=12, model_name="Model"):
    """
    STL-based forecast:
      1) decompose monthly PR into trend + seasonal + residual
      2) fit linear trend on STL trend component
      3) forecast future trend
      4) repeat last 12-month seasonal pattern
      5) add confidence interval from residual std
    """
    series = monthly_series.dropna().copy()

    if len(series) < max(24, seasonal_period * 2):
        raise ValueError(
            f"{model_name}: need at least {max(24, seasonal_period * 2)} monthly points for stable STL forecasting."
        )

    # STL decomposition
    stl = STL(series, period=seasonal_period, robust=True)
    result = stl.fit()

    trend = result.trend.dropna()
    seasonal = result.seasonal
    resid = result.resid.dropna()

    # Fit linear trend to STL trend component
    t = np.arange(len(trend))
    coef = np.polyfit(t, trend.values, 1)
    trend_model = np.poly1d(coef)

    fitted_trend = trend_model(t)

    # Repeat last seasonal cycle for future months
    seasonal_cycle = seasonal.iloc[-seasonal_period:].values
    repeats = int(np.ceil(horizon / seasonal_period))
    future_seasonal = np.tile(seasonal_cycle, repeats)[:horizon]

    # Future dates
    future_dates = pd.date_range(
        start=series.index[-1] + pd.offsets.MonthEnd(1),
        periods=horizon,
        freq="M"
    )

    # Forecast trend + seasonal
    future_t = np.arange(len(trend), len(trend) + horizon)
    future_trend = trend_model(future_t)
    forecast = future_trend + future_seasonal

    # Confidence interval from residual std
    resid_std = resid.std()
    upper = forecast + 2 * resid_std
    lower = forecast - 2 * resid_std

    # In-sample reconstructed fit
    fitted_full = fitted_trend + seasonal.loc[trend.index].values

    degradation_pct_per_year = coef[0] * 12 * 100

    return {
        "model_name": model_name,
        "series": series,
        "trend": trend,
        "seasonal": seasonal,
        "resid": resid,
        "fitted_full": pd.Series(fitted_full, index=trend.index),
        "future_dates": future_dates,
        "future_trend": future_trend,
        "future_seasonal": future_seasonal,
        "forecast": forecast,
        "upper": upper,
        "lower": lower,
        "coef": coef,
        "degradation_pct_per_year": degradation_pct_per_year,
        "resid_std": resid_std
    }


# =========================================================
# RUN FOR MLP AND XGBOOST
# Requires mlp_results["monthly_pr"] and xgb_results["monthly_pr"]
# =========================================================
mlp_stl_fc = stl_forecast_with_ci(
    monthly_series=mlp_results["monthly_pr"],
    horizon=36,
    seasonal_period=12,
    model_name="MLP"
)

xgb_stl_fc = stl_forecast_with_ci(
    monthly_series=xgb_results["monthly_pr"],
    horizon=36,
    seasonal_period=12,
    model_name="XGBoost"
)

print("\n===== STL-BASED DEGRADATION ESTIMATES =====")
print(f"MLP     : {mlp_stl_fc['degradation_pct_per_year']:.4f}% per year")
print(f"XGBoost : {xgb_stl_fc['degradation_pct_per_year']:.4f}% per year")


# =========================================================
# PLOT 1: HISTORICAL MONTHLY PR + STL FORECAST + CI
# =========================================================
plt.figure(figsize=(15, 7))

# Historical monthly PR
plt.plot(
    mlp_stl_fc["series"].index,
    mlp_stl_fc["series"].values,
    alpha=0.30,
    label="MLP Monthly PR"
)
plt.plot(
    xgb_stl_fc["series"].index,
    xgb_stl_fc["series"].values,
    alpha=0.30,
    label="XGBoost Monthly PR"
)

# In-sample fitted series
plt.plot(
    mlp_stl_fc["fitted_full"].index,
    mlp_stl_fc["fitted_full"].values,
    linewidth=2.5,
    label="MLP STL Fit"
)
plt.plot(
    xgb_stl_fc["fitted_full"].index,
    xgb_stl_fc["fitted_full"].values,
    linewidth=2.5,
    label="XGBoost STL Fit"
)

# Forecasts
plt.plot(
    mlp_stl_fc["future_dates"],
    mlp_stl_fc["forecast"],
    linestyle="--",
    linewidth=3,
    label="MLP Forecast (3 yrs)"
)
plt.plot(
    xgb_stl_fc["future_dates"],
    xgb_stl_fc["forecast"],
    linestyle="--",
    linewidth=3,
    label="XGBoost Forecast (3 yrs)"
)

# Confidence intervals
plt.fill_between(
    mlp_stl_fc["future_dates"],
    mlp_stl_fc["lower"],
    mlp_stl_fc["upper"],
    alpha=0.15
)
plt.fill_between(
    xgb_stl_fc["future_dates"],
    xgb_stl_fc["lower"],
    xgb_stl_fc["upper"],
    alpha=0.15
)

plt.axvline(mlp_stl_fc["series"].index[-1], linestyle=":", linewidth=2, color="black", label="Forecast Start")
plt.xlabel("Time")
plt.ylabel("Performance Ratio (PR)")
plt.title("STL-Based 3-Year PR Forecast with Confidence Intervals")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# =========================================================
# PLOT 2: TREND-ONLY COMPARISON
# =========================================================
plt.figure(figsize=(15, 6))

plt.plot(
    mlp_stl_fc["trend"].index,
    mlp_stl_fc["trend"].values,
    linewidth=2.5,
    label=f"MLP Trend ({mlp_stl_fc['degradation_pct_per_year']:.3f}%/yr)"
)
plt.plot(
    xgb_stl_fc["trend"].index,
    xgb_stl_fc["trend"].values,
    linewidth=2.5,
    label=f"XGBoost Trend ({xgb_stl_fc['degradation_pct_per_year']:.3f}%/yr)"
)

plt.plot(
    mlp_stl_fc["future_dates"],
    mlp_stl_fc["future_trend"],
    linestyle="--",
    linewidth=3,
    label="MLP Trend Forecast"
)
plt.plot(
    xgb_stl_fc["future_dates"],
    xgb_stl_fc["future_trend"],
    linestyle="--",
    linewidth=3,
    label="XGBoost Trend Forecast"
)

plt.xlabel("Time")
plt.ylabel("Trend Component of PR")
plt.title("STL Trend Component and 3-Year Trend Forecast")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# =========================================================
# PLOT 3: STL DECOMPOSITION FOR MLP
# =========================================================
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(mlp_stl_fc["series"])
axes[0].set_title("MLP Monthly PR")
axes[0].grid(True)

axes[1].plot(mlp_stl_fc["trend"])
axes[1].set_title("MLP Trend Component")
axes[1].grid(True)

axes[2].plot(mlp_stl_fc["seasonal"])
axes[2].set_title("MLP Seasonal Component")
axes[2].grid(True)

axes[3].plot(mlp_stl_fc["resid"])
axes[3].set_title("MLP Residual Component")
axes[3].grid(True)

plt.tight_layout()
plt.show()


# =========================================================
# PLOT 4: STL DECOMPOSITION FOR XGBOOST
# =========================================================
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(xgb_stl_fc["series"])
axes[0].set_title("XGBoost Monthly PR")
axes[0].grid(True)

axes[1].plot(xgb_stl_fc["trend"])
axes[1].set_title("XGBoost Trend Component")
axes[1].grid(True)

axes[2].plot(xgb_stl_fc["seasonal"])
axes[2].set_title("XGBoost Seasonal Component")
axes[2].grid(True)

axes[3].plot(xgb_stl_fc["resid"])
axes[3].set_title("XGBoost Residual Component")
axes[3].grid(True)

plt.tight_layout()
plt.show()


# =========================================================
# SAVE FORECAST TABLE
# =========================================================
forecast_table = pd.DataFrame({
    "MLP_forecast": pd.Series(mlp_stl_fc["forecast"], index=mlp_stl_fc["future_dates"]),
    "MLP_lower_95": pd.Series(mlp_stl_fc["lower"], index=mlp_stl_fc["future_dates"]),
    "MLP_upper_95": pd.Series(mlp_stl_fc["upper"], index=mlp_stl_fc["future_dates"]),
    "XGB_forecast": pd.Series(xgb_stl_fc["forecast"], index=xgb_stl_fc["future_dates"]),
    "XGB_lower_95": pd.Series(xgb_stl_fc["lower"], index=xgb_stl_fc["future_dates"]),
    "XGB_upper_95": pd.Series(xgb_stl_fc["upper"], index=xgb_stl_fc["future_dates"]),
})

forecast_table.to_csv("stl_3year_pr_forecast_with_ci.csv")
print("\nSaved: stl_3year_pr_forecast_with_ci.csv")